# Introduction to Function and Tool Calling

By default, language models emit **text**. **Function calling** (also called **tool calling**) lets them emit **structured actions**: a tool name plus JSON arguments. Your runtime validates those arguments, executes the real function, and feeds the result back to the model.

That bridge — language → structured intent → code execution → observation — is what makes agents **actionable** instead of merely conversational.

This notebook covers:

1. **Terminology** — functions vs tools across providers.
2. **JSON Schema** — the contract between model and runtime.
3. **Tool registry** — one dispatch point for all implementations.
4. **API overview** — how OpenAI-style tool calling works in requests and responses.
5. **Hands-on** — working tools for a **ShopPulse Order API** assistant, including a full offline tool loop.

Everything is self-contained. No prior notebooks or frameworks are required.

In [ ]:
"""
ShopPulse Order API — teaching environment for function/tool calling.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- Documentation chunks (retrieval / search_docs target) ---
API_DOCS: list[dict[str, str]] = [
    {
        "id": "auth-01",
        "title": "API authentication",
        "text": "Send X-ShopPulse-Key header with every request. Keys are scoped to a store. Rotate from Dashboard > Developers.",
    },
    {
        "id": "orders-02",
        "title": "Order lifecycle",
        "text": "Orders move: pending → paid → shipped → delivered. Cancellations allowed only in pending or paid within 1 hour.",
    },
    {
        "id": "refunds-03",
        "title": "Refund policy",
        "text": "Full refunds within 30 days of delivery. Partial refunds require manager approval. Refund API: POST /v1/orders/{id}/refunds.",
    },
    {
        "id": "shipping-04",
        "title": "Shipping updates",
        "text": "PATCH /v1/orders/{id}/shipping with tracking_number and carrier. Customers receive email on update.",
    },
]

# --- Live order store (tool side effects) ---
ORDERS: dict[str, dict[str, Any]] = {
    "ORD-7001": {
        "id": "ORD-7001",
        "customer_email": "alice@example.com",
        "status": "paid",
        "total_usd": 89.99,
        "items": ["Wireless earbuds"],
        "delivered_at": None,
        "shipping": {"carrier": None, "tracking_number": None},
    },
    "ORD-7002": {
        "id": "ORD-7002",
        "customer_email": "bob@example.com",
        "status": "delivered",
        "total_usd": 24.50,
        "items": ["USB-C cable"],
        "delivered_at": "2026-06-15",
        "shipping": {"carrier": "FedEx", "tracking_number": "FX123"},
    },
}

REFUNDS: list[dict[str, Any]] = []

print(f"API docs: {len(API_DOCS)} | Orders: {len(ORDERS)}")

---

## 1. Functions vs Tools — Terminology

Vendors use overlapping terms. In practice:

| Term | Who uses it | Meaning |
|------|-------------|--------|
| **Function calling** | OpenAI (older docs) | Model returns a call targeting a named function with JSON args |
| **Tool calling** | OpenAI (current), Anthropic, Gemini | Model picks among `tools[]`; functions are one tool type |
| **`@tool` decorator** | LangChain and similar | Python function + auto-generated schema |
| **MCP tool** | Model Context Protocol | Standardized remote tool discovered over a wire protocol |

In this notebook, **tool** is the umbrella concept; **function** is a JSON-schema-described callable the model can invoke. The model sees schemas; your code owns implementations.

---

## 2. Why Models Need Structured Actions

Free-text commands are ambiguous:

```
Model text: "Issue a refund for order ORD-7001"
  → Was that executed? For what amount? Who authorized it?
```

Structured tool call:

```json
{
  "name": "create_refund",
  "arguments": {"order_id": "ORD-7001", "amount_usd": 89.99, "reason": "customer request"}
}
```

Your runtime can **validate** arguments, **check authorization**, **execute** with timeouts, and return a structured **observation** — the foundation of reliable agent behavior.

---

## 3. JSON Schema as the Contract

OpenAI-compatible APIs accept tools defined with **JSON Schema** for parameters:

| Field | Purpose |
|-------|--------|
| `name` | Identifier the model must emit |
| `description` | When to use the tool — the most important field for reliability |
| `parameters.type: object` | Arguments are key-value pairs |
| `properties` | Allowed parameter names and types |
| `required` | Parameters the model must always fill |
| `enum` | Closed set of allowed values |

Bad schemas cause wrong tools, missing arguments, or the model refusing to call anything at all.

In [ ]:
SEARCH_DOCS_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_docs",
        "description": (
            "Search ShopPulse API documentation by keyword. "
            "Use for policy, auth, and how-to questions — not for live order data."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Short search phrase, e.g. 'refund policy' or 'API key'.",
                },
                "limit": {
                    "type": "integer",
                    "description": "Max documentation chunks to return (1-5).",
                },
            },
            "required": ["query"],
        },
    },
}

GET_ORDER_SCHEMA = {
    "type": "function",
    "function": {
        "name": "get_order",
        "description": "Fetch live order details by order ID. Use when the user mentions a specific order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "Order ID like ORD-7001.",
                },
            },
            "required": ["order_id"],
        },
    },
}

CREATE_REFUND_SCHEMA = {
    "type": "function",
    "function": {
        "name": "create_refund",
        "description": (
            "Create a refund for a delivered order within 30-day policy. "
            "Do not use for pending orders — use cancel instead."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount_usd": {"type": "number", "description": "Refund amount in USD."},
                "reason": {"type": "string", "description": "Brief reason for audit log."},
            },
            "required": ["order_id", "amount_usd", "reason"],
        },
    },
}

UPDATE_SHIPPING_SCHEMA = {
    "type": "function",
    "function": {
        "name": "update_shipping",
        "description": "Add tracking number and carrier to a paid order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "carrier": {"type": "string", "enum": ["FedEx", "UPS", "USPS", "DHL"]},
                "tracking_number": {"type": "string"},
            },
            "required": ["order_id", "carrier", "tracking_number"],
        },
    },
}

ALL_TOOL_SCHEMAS = [
    SEARCH_DOCS_SCHEMA,
    GET_ORDER_SCHEMA,
    CREATE_REFUND_SCHEMA,
    UPDATE_SHIPPING_SCHEMA,
]

print("Tool schemas defined:")
for schema in ALL_TOOL_SCHEMAS:
    print(f"  - {schema['function']['name']}")

---

## 4. Tool Implementations and Registry

The model never calls Python directly. You register **implementations** behind each schema name:

```
         ┌─────────────────────────────────┐
         │         Tool registry            │
 schemas │  search_docs    → search_docs()  │
 ───────►│  get_order      → get_order()    │
         │  create_refund  → create_refund()│
         └───────────────┬─────────────────┘
                         │ dispatch by name
                         ▼
                   Python / HTTP / database
```

In [ ]:
def search_docs(query: str, limit: int = 3) -> str:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    scored = []
    for doc in API_DOCS:
        text = f"{doc['title']} {doc['text']}".lower()
        score = sum(1 for term in terms if term in text) if terms else 1
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    hits = [d for _, d in scored[:max(1, min(limit, 5))]]
    return "\n".join(f"[{h['id']}] {h['title']}: {h['text']}" for h in hits)


def get_order(order_id: str) -> dict[str, Any]:
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"error": f"Order {order_id} not found"}
    return order


def create_refund(order_id: str, amount_usd: float, reason: str) -> dict[str, Any]:
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"success": False, "error": "order not found"}
    if order["status"] != "delivered":
        return {"success": False, "error": f"cannot refund order in status '{order['status']}'"}
    if amount_usd > order["total_usd"]:
        return {"success": False, "error": "amount exceeds order total"}
    refund = {
        "id": f"REF-{uuid.uuid4().hex[:6].upper()}",
        "order_id": order_id.upper(),
        "amount_usd": amount_usd,
        "reason": reason,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    REFUNDS.append(refund)
    return {"success": True, "refund": refund}


def update_shipping(order_id: str, carrier: str, tracking_number: str) -> dict[str, Any]:
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"success": False, "error": "order not found"}
    if order["status"] not in ("paid", "shipped"):
        return {"success": False, "error": f"cannot ship order in status '{order['status']}'"}
    order["status"] = "shipped"
    order["shipping"] = {"carrier": carrier, "tracking_number": tracking_number}
    return {"success": True, "order": order}


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "search_docs": search_docs,
    "get_order": get_order,
    "create_refund": create_refund,
    "update_shipping": update_shipping,
}

# Sanity checks
print(search_docs("refund policy")[:100], "...")
print(get_order("ORD-7001"))
print(create_refund("ORD-7002", 24.50, "customer request"))

---

## 5. OpenAI Tools API — Request and Response Shape

A typical Chat Completions request with tools:

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[...],
    tools=ALL_TOOL_SCHEMAS,
    tool_choice="auto",  # "none" | "auto" | force specific tool
)
```

The assistant message may include `tool_calls`:

```json
{
  "role": "assistant",
  "content": null,
  "tool_calls": [{
    "id": "call_abc123",
    "type": "function",
    "function": {
      "name": "get_order",
      "arguments": "{\"order_id\": \"ORD-7001\"}"
    }
  }]
}
```

You execute each call, then append a `role: tool` message with `tool_call_id` and the result, then **call the API again** until the model returns text without tool calls.

---

## 6. Dispatching Tool Calls Safely

| Step | Responsibility |
|------|----------------|
| **Parse** | `json.loads(arguments)` — handle malformed JSON |
| **Validate** | Check required fields and types against schema |
| **Authorize** | User scopes, rate limits, HITL for sensitive tools |
| **Execute** | Call registry with timeout |
| **Serialize** | Return string or JSON for the `tool` message |

In [ ]:
REQUIRED_FIELDS: dict[str, list[str]] = {
    "search_docs": ["query"],
    "get_order": ["order_id"],
    "create_refund": ["order_id", "amount_usd", "reason"],
    "update_shipping": ["order_id", "carrier", "tracking_number"],
}

SENSITIVE_TOOLS = {"create_refund"}


def validate_args(name: str, args: dict[str, Any]) -> tuple[bool, str]:
    required = REQUIRED_FIELDS.get(name, [])
    missing = [f for f in required if f not in args]
    if missing:
        return False, f"missing required fields: {missing}"
    if name == "update_shipping" and args.get("carrier") not in ["FedEx", "UPS", "USPS", "DHL"]:
        return False, "invalid carrier enum"
    return True, "ok"


def dispatch_tool_call(
    tool_call: dict[str, Any],
    *,
    allow_sensitive: bool = False,
) -> dict[str, Any]:
    """Execute one tool call and return the tool message for the conversation."""
    fn = tool_call["function"]
    name = fn["name"]
    call_id = tool_call["id"]

    if name not in TOOL_REGISTRY:
        return {"role": "tool", "tool_call_id": call_id, "content": json.dumps({"error": f"unknown tool: {name}"})}

    if name in SENSITIVE_TOOLS and not allow_sensitive:
        return {"role": "tool", "tool_call_id": call_id, "content": json.dumps({"error": "requires human approval"})}

    try:
        args = json.loads(fn["arguments"])
    except json.JSONDecodeError as exc:
        return {"role": "tool", "tool_call_id": call_id, "content": json.dumps({"error": f"invalid JSON: {exc}"})}

    ok, msg = validate_args(name, args)
    if not ok:
        return {"role": "tool", "tool_call_id": call_id, "content": json.dumps({"error": msg})}

    try:
        result = TOOL_REGISTRY[name](**args)
        content = result if isinstance(result, str) else json.dumps(result)
    except Exception as exc:
        content = json.dumps({"error": str(exc)})

    return {"role": "tool", "tool_call_id": call_id, "content": content}


# Demo dispatch
sample_call = {
    "id": "call_demo_1",
    "type": "function",
    "function": {
        "name": "get_order",
        "arguments": json.dumps({"order_id": "ORD-7001"}),
    },
}
tool_message = dispatch_tool_call(sample_call)
print(pretty(tool_message))

---

## 7. `tool_choice` Modes

| Value | Behavior | When to use |
|-------|----------|-------------|
| `"auto"` | Model may answer in text or call tools | Default for assistants |
| `"none"` | Disable tool calls | Pure chat / safety mode |
| `{"type":"function","function":{"name":"search_docs"}}` | Force a specific tool | Pipelines where step 1 must always search |

In [ ]:
TOOL_CHOICE_MODES = {
    "auto": "Model decides: text answer or tool call",
    "none": "Text only — tools disabled",
    "forced_get_order": {
        "type": "function",
        "function": {"name": "get_order"},
    },
}

for mode, desc in TOOL_CHOICE_MODES.items():
    print(f"{mode}: {desc if isinstance(desc, str) else pretty(desc)}")

---

## 8. Bad Schema vs Good Schema

Vague names and empty descriptions cause **wrong-tool** or **no-tool** behavior.

In [ ]:
BAD_SCHEMA = {
    "type": "function",
    "function": {
        "name": "helper",
        "description": "Helps with stuff",
        "parameters": {"type": "object", "properties": {}},
    },
}

COMPARISON = [
    ("Name", BAD_SCHEMA["function"]["name"], SEARCH_DOCS_SCHEMA["function"]["name"]),
    ("Description length", len(BAD_SCHEMA["function"]["description"]), len(SEARCH_DOCS_SCHEMA["function"]["description"])),
    ("Required params", 0, len(SEARCH_DOCS_SCHEMA["function"]["parameters"]["required"])),
]

print(f"{'Field':<20} {'Bad':<15} {'Good'}")
print("-" * 50)
for field, bad, good in COMPARISON:
    print(f"{field:<20} {str(bad):<15} {good}")

---

## 9. Offline Model Simulator + Full Tool Loop

The `OfflineToolPlanner` mimics what an LLM returns — tool calls or final text — so you can run the **complete loop** without an API key.

In [ ]:
class OfflineToolPlanner:
    """Rule-based stand-in for model tool-calling decisions."""

    def plan(self, user_text: str, messages: list[dict[str, Any]]) -> dict[str, Any]:
        t = user_text.lower()

        # If we already have tool results, synthesize final answer
        tool_msgs = [m for m in messages if m.get("role") == "tool"]
        if tool_msgs:
            return {"role": "assistant", "content": self._finalize(tool_msgs, user_text)}

        order_match = re.search(r"ord-\d+", t, re.IGNORECASE)
        if order_match and "refund" in t:
            oid = order_match.group(0).upper()
            return self._tool_msg("create_refund", {
                "order_id": oid, "amount_usd": 24.50, "reason": "customer request",
            })
        if order_match and any(w in t for w in ("track", "shipping", "carrier")):
            return self._tool_msg("update_shipping", {
                "order_id": order_match.group(0).upper(),
                "carrier": "FedEx", "tracking_number": "FX999888",
            })
        if order_match:
            return self._tool_msg("get_order", {"order_id": order_match.group(0).upper()})
        if any(w in t for w in ("policy", "api key", "auth", "how do", "refund rules")):
            return self._tool_msg("search_docs", {"query": user_text, "limit": 2})
        return {"role": "assistant", "content": "Ask about orders (ORD-####) or ShopPulse API docs."}

    def _tool_msg(self, name: str, args: dict[str, Any]) -> dict[str, Any]:
        return {
            "role": "assistant",
            "content": None,
            "tool_calls": [{
                "id": f"call_{uuid.uuid4().hex[:8]}",
                "type": "function",
                "function": {"name": name, "arguments": json.dumps(args)},
            }],
        }

    def _finalize(self, tool_msgs: list[dict[str, Any]], goal: str) -> str:
        last = json.loads(tool_msgs[-1]["content"])
        if "error" in last:
            return f"Could not complete: {last['error']}"
        if "refund" in goal.lower() and last.get("success"):
            r = last["refund"]
            return f"Refund {r['id']} created for {r['amount_usd']} USD."
        if isinstance(last, dict) and "id" in last and "status" in last:
            return f"Order {last['id']}: status={last['status']}, total=${last['total_usd']}"
        if isinstance(last, str):
            return f"From documentation:\n{last[:300]}"
        return pretty(last)


@dataclass
class ShopPulseToolAssistant:
    """Full tool-calling loop: plan → dispatch → finalize."""

    planner: OfflineToolPlanner = field(default_factory=OfflineToolPlanner)
    messages: list[dict[str, Any]] = field(default_factory=list)
    trace: list[str] = field(default_factory=list)
    max_rounds: int = 4
    allow_sensitive: bool = False

    def run(self, user_message: str) -> str:
        self.messages = [
            {"role": "system", "content": "You are a ShopPulse order API assistant. Use tools for live data and docs."},
            {"role": "user", "content": user_message},
        ]
        self.trace = []

        for round_num in range(self.max_rounds):
            assistant_msg = self.planner.plan(user_message, self.messages)
            self.messages.append(assistant_msg)

            if assistant_msg.get("content") and not assistant_msg.get("tool_calls"):
                self.trace.append(f"round {round_num}: final text")
                return assistant_msg["content"]

            for tc in assistant_msg.get("tool_calls", []):
                name = tc["function"]["name"]
                self.trace.append(f"round {round_num}: tool_call {name}")
                tool_msg = dispatch_tool_call(tc, allow_sensitive=self.allow_sensitive)
                self.messages.append(tool_msg)
                self.trace.append(f"round {round_num}: observation {tool_msg['content'][:80]}...")

        return "Stopped: max_rounds reached"


def demo_query(label: str, query: str, allow_sensitive: bool = False) -> None:
    print("=" * 72)
    print(f"{label}: {query}")
    assistant = ShopPulseToolAssistant(allow_sensitive=allow_sensitive)
    answer = assistant.run(query)
    print(f"Answer: {answer}")
    print(f"Trace: {assistant.trace}")
    print()

In [ ]:
demo_query("Doc search", "What is the refund policy for delivered orders?")
demo_query("Order lookup", "What's the status of order ORD-7001?")
demo_query("Refund blocked", "Refund order ORD-7002 for customer request")
demo_query("Refund allowed", "Refund order ORD-7002 for customer request", allow_sensitive=True)

---

## 10. Provider Comparison — Same Idea, Different Field Names

All major providers follow the same pattern: declare tools → model returns structured calls → client executes → results fed back.

| Provider | Tools in request | Call in response | Result message |
|----------|------------------|------------------|----------------|
| **OpenAI** | `tools=[...]` | `message.tool_calls` | `role: tool`, `tool_call_id` |
| **Anthropic** | `tools=[...]` | `content[].type == "tool_use"` | `tool_result` block |
| **Google Gemini** | `tools` / function declarations | `functionCall` in parts | `functionResponse` |

Your **tool registry and validation logic** can stay provider-agnostic; only the message adapter layer changes.

---

## 11. Generating Schemas from Pydantic

Teams often define arguments once in Pydantic and export JSON Schema to avoid drift between Python signatures and tool definitions.

In [ ]:
try:
    from pydantic import BaseModel, Field

    class SearchDocsArgs(BaseModel):
        query: str = Field(description="Keyword search for API documentation")
        limit: int = Field(default=3, ge=1, le=5)

    class CreateRefundArgs(BaseModel):
        order_id: str
        amount_usd: float = Field(gt=0)
        reason: str = Field(max_length=200)

    print("SearchDocsArgs JSON Schema:")
    print(pretty(SearchDocsArgs.model_json_schema()))
except ImportError:
    print("Pydantic not installed — schemas defined manually in section 3 still work.")

---

## 12. MCP — Remote Tool Discovery (Preview)

The **Model Context Protocol (MCP)** lets agents discover tools from remote servers instead of hard-coding a local registry. The mental model:

```
Local registry (this notebook)     MCP (production scale)
        │                                    │
   TOOL_REGISTRY                    remote server advertises
   defined in code                  tool manifests at runtime
```

Your dispatch pattern stays the same — only the **source of schemas** changes from static code to dynamic discovery.

In [ ]:
LOCAL_TOOLS = list(TOOL_REGISTRY.keys())

MCP_DISCOVERED = [
    {"name": "search_docs", "server": "shoppulse-docs-mcp", "description": "API documentation search"},
    {"name": "github_pr_status", "server": "engineering-mcp", "description": "PR checks from GitHub"},
]

print("Local registry:", LOCAL_TOOLS)
print("MCP discovered:", [t["name"] for t in MCP_DISCOVERED])

---

## 13. Design Checklist

1. **One tool = one verb** — `get_order`, not `handle_everything`.
2. **Descriptions say when NOT to use** the tool — reduces wrong-tool errors.
3. **Use `enum`** for small closed sets (carriers, environments).
4. **Registry is the single dispatch point** — never scatter `if name ==` across the codebase.
5. **Log** `tool_call_id`, latency, and redacted arguments for audit.
6. **Sensitive tools** require explicit approval flag or HITL gate.
7. **Return structured errors** in tool messages so the model can retry with fixed args.

---

## 14. Operations That Should Never Be Unguarded Tools

For ShopPulse (and most production APIs), these require human approval or workflow gates — not autonomous tool execution:

- `delete_order` / `purge_customer_data`
- `create_refund` above policy threshold without review
- `rotate_api_key` / `change_billing_plan`
- Arbitrary SQL or shell execution

In [ ]:
BLOCKED_WITHOUT_HITL = {
    "delete_order", "purge_customer_data", "rotate_api_key",
    "change_billing_plan", "execute_sql", "run_shell",
}

APPROVAL_REQUIRED = {"create_refund"}  # allowed with allow_sensitive=True in our demo

def classify_tool_risk(name: str) -> str:
    if name in BLOCKED_WITHOUT_HITL:
        return "NEVER autonomous"
    if name in APPROVAL_REQUIRED:
        return "HITL or explicit flag"
    return "autonomous OK with validation"


for tool in list(TOOL_REGISTRY.keys()) + ["delete_order", "rotate_api_key"]:
    print(f"{tool:22} → {classify_tool_risk(tool)}")

---

## 15. Optional — Live OpenAI Tool Calling

Set `USE_LIVE_LLM = True` with a valid API key to run a real tool-calling request.

In [ ]:
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    try:
        from openai import OpenAI
        client = OpenAI()
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Use search_docs for ShopPulse API policy questions."},
                {"role": "user", "content": "What is the refund policy?"},
            ],
            tools=ALL_TOOL_SCHEMAS,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        if msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"Tool call: {tc.function.name}({tc.function.arguments})")
                tool_msg = dispatch_tool_call({
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                })
                print("Result:", tool_msg["content"][:200])
        else:
            print(msg.content)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print("Offline mode — use ShopPulseToolAssistant demos above.")
    print("Tool schemas ready:", [s["function"]["name"] for s in ALL_TOOL_SCHEMAS])

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Vague tool names (`helper`) | Wrong tool selected | Specific verbs + descriptions |
| No argument validation | Bad side effects | Validate before registry dispatch |
| Model executes tools directly | Security hole | Runtime always dispatches |
| Skipping `tool_call_id` in response | API errors on follow-up | Match IDs exactly |
| One mega-tool | Unreliable args | Split by domain |
| Sensitive tools without HITL | Financial / data harm | Approval gates |

---

## 17. Check Your Understanding

1. What is the difference between **function calling** and **tool calling** in modern OpenAI APIs?
2. Why does the model see **schemas** but not Python function bodies?
3. What fields in a JSON Schema most improve tool selection reliability?
4. Walk through the five steps in `dispatch_tool_call`.
5. What does `tool_choice="none"` do?
6. Why was the refund demo blocked until `allow_sensitive=True`?
7. How does MCP differ from a local `TOOL_REGISTRY`?

---

## 18. Summary

**Tool calling** lets models emit structured `{name, arguments}` instead of only text. Your runtime **parses, validates, authorizes, executes**, and returns observations — the core bridge between language and systems.

**Key takeaways:**

- **Tools** are JSON-schema contracts; **implementations** live in a **registry** you control.
- **Descriptions** and **`required` fields** are the main levers for reliable tool use.
- The API loop is: user message → assistant `tool_calls` → `role: tool` results → call model again → final text.
- **`tool_choice`** forces, disables, or auto-selects tools.
- **Validate and gate** sensitive tools — refunds, deletes, and credential changes need HITL.
- **Provider adapters** differ in field names; registry logic stays the same.
- **`ShopPulseToolAssistant`** demonstrates the full offline loop you can swap to a live LLM planner.

You can now define tools for any API: write the schema, implement the function, register it, and dispatch safely.